In [ ]:
# =========================
# nb_ingest_pershing_v2
# Stage 1: Faithful Source Ingestion
# =========================
"""
Purpose: Provide a faithful representation of Pershing source files
         in Fabric for Stage 2 processing.

File Structure:
- Row 1: Column headers
- Row 2+: Data
"""

# ---------- Establish Workspace Context ----------
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "pershing"
dfm_name = "Pershing"

print(f"[INFO] Starting Pershing ingestion for period={period}, run_id={run_id}")

In [ ]:
# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
from datetime import datetime

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
print(f"[INFO] Landing path: {landing_path}")

In [ ]:
# ---------- Helper Functions ----------

def normalize_column_name(col_name):
    """Normalize column names: remove special chars, lowercase, replace spaces with underscores."""
    import re
    normalized = str(col_name).strip().lower()
    normalized = re.sub(r'[^\w\s]', '', normalized)
    normalized = re.sub(r'\s+', '_', normalized)
    return normalized

def read_csv_standard(file_path, source_file_name):
    """
    Read standard CSV file where:
    - Row 1 = Headers
    - Row 2+ = Data
    
    Returns: Spark DataFrame with all columns as strings
    """
    try:
        # Read with pandas, standard format
        pdf = pd.read_csv(
            file_path,
            dtype=str,   # Keep everything as string for fidelity
            encoding='utf-8',
            keep_default_na=False,  # Preserve empty strings
            na_filter=False
        )
        
        # Normalize column names
        pdf.columns = [normalize_column_name(col) for col in pdf.columns]
        
        # Remove any completely empty rows
        pdf = pdf[pdf.apply(lambda x: x.str.strip().str.len().sum(), axis=1) > 0]
        
        if len(pdf) == 0:
            print(f"[WARNING] No data rows found in {source_file_name}")
            return None
        
        # Convert to Spark DataFrame
        sdf = spark.createDataFrame(pdf)
        
        # Add source tracking
        sdf = sdf.withColumn("source_file", F.lit(source_file_name))
        
        print(f"[INFO] Successfully read {source_file_name}: {sdf.count()} rows, {len(sdf.columns)-1} columns")
        return sdf
        
    except Exception as e:
        print(f"[ERROR] Failed to read {source_file_name}: {str(e)}")
        return None

def add_row_identifiers(df, file_prefix):
    """Add unique row identifiers to each record."""
    w = Window.orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("_row_num", F.row_number().over(w))
          .withColumn(
              "source_row_id",
              F.concat_ws(":", 
                  F.col("source_file"),
                  F.lit(file_prefix),
                  F.col("_row_num").cast("string")
              )
          )
          .drop("_row_num")
    )

def add_metadata(df):
    """Add standard metadata columns to the dataframe."""
    return (
        df.withColumn("period", F.lit(period))
          .withColumn("run_id", F.lit(run_id))
          .withColumn("dfm_id", F.lit(dfm_id))
          .withColumn("dfm_name", F.lit(dfm_name))
          .withColumn("ingested_at", F.current_timestamp())
    )

print("[INFO] Helper functions loaded")

In [ ]:
# ---------- Discover Files ----------
print("[STEP 1] Discovering source files...")

try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    csv_files = [f for f in entries if f.name.lower().endswith(".csv")]
    print(f"[INFO] Found {len(csv_files)} CSV file(s)")
    for f in csv_files:
        print(f"  - {f.name}")
except Exception as e:
    print(f"[ERROR] Could not access landing path: {str(e)}")
    mssparkutils.notebook.exit("NO_FILES")

if len(csv_files) == 0:
    print("[ERROR] No CSV files found")
    mssparkutils.notebook.exit("NO_FILES")

In [ ]:
# ---------- Identify File Types ----------
print("[STEP 2] Categorizing files...")

# Pershing can have multiple file types
positions_files = [f for f in csv_files if "positions" in f.name.lower()]
psl_valuation_files = [f for f in csv_files if "psl_valuation" in f.name.lower() or "valuation_holdings" in f.name.lower()]
other_files = [f for f in csv_files if f not in positions_files and f not in psl_valuation_files]

print(f"[INFO] Positions files: {len(positions_files)}")
for f in positions_files:
    print(f"  - {f.name}")

print(f"[INFO] PSL Valuation files: {len(psl_valuation_files)}")
for f in psl_valuation_files:
    print(f"  - {f.name}")

if other_files:
    print(f"[INFO] Other files: {len(other_files)}")
    for f in other_files:
        print(f"  - {f.name}")

In [ ]:
# ---------- Read Positions Files ----------
print("[STEP 3] Reading Positions files...")

df_positions_list = []

for pos_file in positions_files:
    df = read_csv_standard(pos_file.path, pos_file.name)
    if df:
        df = add_row_identifiers(df, "POS")
        df = df.withColumn("file_type", F.lit("POSITIONS"))
        df_positions_list.append(df)
        print(f"[INFO] Processed {pos_file.name}: {df.count()} rows")

# Combine all positions DataFrames
df_positions = None
if df_positions_list:
    if len(df_positions_list) == 1:
        df_positions = df_positions_list[0]
    else:
        # Union all positions files - they should have the same schema
        df_positions = df_positions_list[0]
        for df in df_positions_list[1:]:
            df_positions = df_positions.unionByName(df, allowMissingColumns=True)
    
    print(f"[INFO] Total positions rows: {df_positions.count()}")
    print(f"[INFO] Positions columns: {', '.join(sorted([c for c in df_positions.columns if c not in ['source_file', 'source_row_id', 'file_type']]))}")
else:
    print("[INFO] No positions files processed")

In [ ]:
# ---------- Read PSL Valuation Files ----------
print("[STEP 4] Reading PSL Valuation files...")

df_psl_valuation_list = []

for psl_file in psl_valuation_files:
    df = read_csv_standard(psl_file.path, psl_file.name)
    if df:
        df = add_row_identifiers(df, "PSL")
        df = df.withColumn("file_type", F.lit("PSL_VALUATION"))
        df_psl_valuation_list.append(df)
        print(f"[INFO] Processed {psl_file.name}: {df.count()} rows")

# Combine all PSL valuation DataFrames
df_psl_valuation = None
if df_psl_valuation_list:
    if len(df_psl_valuation_list) == 1:
        df_psl_valuation = df_psl_valuation_list[0]
    else:
        # Union all PSL files - they should have the same schema
        df_psl_valuation = df_psl_valuation_list[0]
        for df in df_psl_valuation_list[1:]:
            df_psl_valuation = df_psl_valuation.unionByName(df, allowMissingColumns=True)
    
    print(f"[INFO] Total PSL valuation rows: {df_psl_valuation.count()}")
    print(f"[INFO] PSL valuation columns: {', '.join(sorted([c for c in df_psl_valuation.columns if c not in ['source_file', 'source_row_id', 'file_type']]))}")
else:
    print("[INFO] No PSL valuation files processed")

In [ ]:
# ---------- Read Other Files ----------
print("[STEP 5] Reading other files...")

df_other_list = []

for other_file in other_files:
    df = read_csv_standard(other_file.path, other_file.name)
    if df:
        df = add_row_identifiers(df, "OTHER")
        df = df.withColumn("file_type", F.lit("OTHER"))
        df_other_list.append(df)
        print(f"[INFO] Processed {other_file.name}: {df.count()} rows")

# Store each "other" file separately if needed
df_other = None
if df_other_list:
    if len(df_other_list) == 1:
        df_other = df_other_list[0]
    else:
        df_other = df_other_list[0]
        for df in df_other_list[1:]:
            df_other = df_other.unionByName(df, allowMissingColumns=True)
    
    print(f"[INFO] Total other rows: {df_other.count()}")
else:
    print("[INFO] No other files processed")

In [ ]:
# ---------- Profile the Data ----------
print("\n" + "=" * 80)
print("[DATA PROFILE] Source File Statistics")
print("=" * 80)

if df_positions:
    print(f"\nPOSITIONS FILES: {len(positions_files)} file(s)")
    print(f"  Total rows: {df_positions.count()}")
    print(f"  Total columns: {len([c for c in df_positions.columns if c not in ['source_file', 'source_row_id', 'file_type']])}")
    
    # Check for key columns
    if 'accountnumber' in df_positions.columns:
        distinct_accounts = df_positions.select("accountnumber").distinct().count()
        null_accounts = df_positions.filter(F.col("accountnumber").isNull() | (F.trim(F.col("accountnumber")) == "")).count()
        print(f"  Distinct account numbers: {distinct_accounts}")
        print(f"  Rows with empty account number: {null_accounts}")
    
    if 'isin' in df_positions.columns:
        distinct_isins = df_positions.select("isin").filter(F.col("isin") != "").distinct().count()
        print(f"  Distinct ISIN codes (non-empty): {distinct_isins}")

if df_psl_valuation:
    print(f"\nPSL VALUATION FILES: {len(psl_valuation_files)} file(s)")
    print(f"  Total rows: {df_psl_valuation.count()}")
    print(f"  Total columns: {len([c for c in df_psl_valuation.columns if c not in ['source_file', 'source_row_id', 'file_type']])}")
    
    # Check for key columns
    if 'pslaccountreference' in df_psl_valuation.columns:
        distinct_accounts = df_psl_valuation.select("pslaccountreference").distinct().count()
        null_accounts = df_psl_valuation.filter(F.col("pslaccountreference").isNull() | (F.trim(F.col("pslaccountreference")) == "")).count()
        print(f"  Distinct PSL account references: {distinct_accounts}")
        print(f"  Rows with empty PSL account reference: {null_accounts}")
    
    if 'isin' in df_psl_valuation.columns:
        distinct_isins = df_psl_valuation.select("isin").filter(F.col("isin") != "").distinct().count()
        print(f"  Distinct ISIN codes (non-empty): {distinct_isins}")

if df_other:
    print(f"\nOTHER FILES: {len(other_files)} file(s)")
    print(f"  Total rows: {df_other.count()}")

print("=" * 80 + "\n")

In [ ]:
# ---------- Add Metadata Columns ----------
print("[STEP 6] Adding metadata columns...")

rows_ready = 0

if df_positions:
    df_positions = add_metadata(df_positions)
    pos_count = df_positions.count()
    rows_ready += pos_count
    print(f"[INFO] Metadata added to positions: {pos_count} rows ready")

if df_psl_valuation:
    df_psl_valuation = add_metadata(df_psl_valuation)
    psl_count = df_psl_valuation.count()
    rows_ready += psl_count
    print(f"[INFO] Metadata added to PSL valuation: {psl_count} rows ready")

if df_other:
    df_other = add_metadata(df_other)
    other_count = df_other.count()
    rows_ready += other_count
    print(f"[INFO] Metadata added to other files: {other_count} rows ready")

print(f"[INFO] Total rows ready for staging: {rows_ready}")

In [ ]:
# ---------- Write to Staging Tables ----------
print("[STEP 7] Writing to staging tables...")

rows_written = 0

if df_positions:
    table_name = "stage1_pershing_positions_raw"
    print(f"[INFO] Writing positions to {table_name}...")
    df_positions.write.mode("append").saveAsTable(table_name)
    pos_count = df_positions.count()
    rows_written += pos_count
    print(f"[SUCCESS] Written {pos_count} position rows")

if df_psl_valuation:
    table_name = "stage1_pershing_psl_valuation_raw"
    print(f"[INFO] Writing PSL valuation to {table_name}...")
    df_psl_valuation.write.mode("append").saveAsTable(table_name)
    psl_count = df_psl_valuation.count()
    rows_written += psl_count
    print(f"[SUCCESS] Written {psl_count} PSL valuation rows")

if df_other:
    table_name = "stage1_pershing_other_raw"
    print(f"[INFO] Writing other files to {table_name}...")
    df_other.write.mode("append").saveAsTable(table_name)
    other_count = df_other.count()
    rows_written += other_count
    print(f"[SUCCESS] Written {other_count} other rows")

print(f"[INFO] Total rows written: {rows_written}")

In [ ]:
# ---------- Write Audit Log ----------
print("[STEP 8] Writing audit log...")

try:
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, 
         parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{run_id}', '{period}', '{dfm_id}', {len(csv_files)}, {rows_written}, 
         0, 0, 'OK', current_timestamp(), current_timestamp())
    """)
    print("[SUCCESS] Audit log written")
except Exception as e:
    print(f"[WARNING] Could not write audit log: {str(e)}")

In [ ]:
# ---------- Summary ----------
print("\n" + "=" * 80)
print("[INGESTION COMPLETE]")
print("=" * 80)
print(f"Period: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")
print(f"Files processed: {len(csv_files)}")
print(f"  - Positions files: {len(positions_files)}")
print(f"  - PSL Valuation files: {len(psl_valuation_files)}")
print(f"  - Other files: {len(other_files)}")
print(f"Total rows ingested: {rows_written}")
if df_positions:
    print(f"  - Positions: {df_positions.count()} rows")
if df_psl_valuation:
    print(f"  - PSL Valuation: {df_psl_valuation.count()} rows")
if df_other:
    print(f"  - Other: {df_other.count()} rows")
print(f"Status: OK")
print("=" * 80)

mssparkutils.notebook.exit("OK")